In [1]:
from src.utils import (
    get_model_and_config,
    keypoint_unscaler,
)
from src.heatmap import extract_keypoints
import torch
import cv2
import inspect
import json
import albumentations as A
from albumentations.pytorch.transforms import ToTensorV2
model_name = "heatmap"
model, cfg = get_model_and_config(model_name)

c:\Users\nebur\AppData\Local\pypoetry\Cache\virtualenvs\keypoint-detection-ukibM8zw-py3.12\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\nebur\AppData\Local\pypoetry\Cache\virtualenvs\keypoint-detection-ukibM8zw-py3.12\Lib\site-packages\albumentations\__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.6'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:
def config_to_dict(config):
    """Convert a config object (or class) to a serializable dictionary."""
    if inspect.isclass(config):
        # Handle class (like BaseConfig)
        attrs = {k: v for k, v in vars(config).items() 
                if not k.startswith('__') and not callable(v)}
    else:
        # Handle instance (like HeatmapConfig)
        attrs = {k: v for k, v in vars(config).items() 
                if not k.startswith('__') and k != 'base' and not callable(v)}
    
    # Handle nested base config
    if hasattr(config, 'base'):
        attrs['__base_config__'] = config_to_dict(config.base)
    
    # Handle non-serializable objects (simplified)
    for k, v in attrs.items():
        if isinstance(v, torch.device):
            attrs[k] = str(v)
        elif isinstance(v, torch.nn.Module):
            attrs[k] = v.__class__.__name__  # Save only the class name
    
    return attrs

# Example usage:
model_name = "heatmap"
config_dict = config_to_dict(cfg)
with open(f'{model_name}_config.json', 'w') as f:
    json.dump(config_dict, f, indent=4)

In [3]:
def dict_to_config(d, model_config, base_config_class=None):
    """Reconstruct a ModelConfig from a dictionary."""
    base_dict = d.pop('__base_config__', None)
    if base_dict and base_config_class:
        base_config = base_config_class()
        for k, v in base_dict.items():
            setattr(base_config, k, v)
    else:
        base_config = None
    
    config = model_config(base_config=base_config_class)
    for k, v in d.items():
        setattr(config, k, v)
    
    return config

# Example reloading:
with open(f'{model_name}_config.json', 'r') as f:
    loaded_dict = json.load(f)

from config import BaseConfig, ModelConfig
loaded_config = dict_to_config(loaded_dict, ModelConfig, BaseConfig)


loaded_config.learn_rate

0.0005

In [2]:
class CourtLineDetector:
    def __init__(self, cfg, model, load_model):
        self.model = model
        self.model.load_state_dict(torch.load(load_model, map_location='cpu'))
        self.model = self.model.to(cfg.device)
        self.model.eval()
        self.cfg = cfg
        self.width = cfg.width
        self.height = cfg.height
        
        self.transform = A.Compose([
            A.Resize(width=cfg.width, height=cfg.height),
            A.Normalize(mean=cfg.mean, std=cfg.std),
            ToTensorV2(p=1.0),
        ])

    def predict(self, image_path):
        image = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
        original_h, original_w = image.shape[:2]
        image_tensor = self.transform(image=image)["image"].unsqueeze(0)

        self.model.eval()
        with torch.no_grad():
            outputs = self.model(image_tensor.to(self.cfg.device))

        if self.cfg.heatmap:
            keypoints, max_values = extract_keypoints(outputs)
            keypoints = keypoints.squeeze().cpu().numpy()
        else:
            keypoints = outputs.squeeze().cpu().numpy()


        keypoints = keypoint_unscaler(
            cfg, keypoints, original_w, original_h
        )
        return keypoints, max_values

    def draw_keypoints(self, image_path, keypoints):
        if keypoints.ndim == 2:
            keypoints = keypoints.flatten()

        image = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
        image = image.copy()
        for i in range(0, len(keypoints), 2):
            x = int(keypoints[i])
            y = int(keypoints[i+1])
            cv2.putText(image, str(i//2), (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
            cv2.circle(image, (x, y), 5, (0, 0, 255), -1)
        return image
    
image_path = cfg.sample_image_path[0]
orig_image = cv2.imread(image_path)
orig_image = cv2.cvtColor(orig_image, cv2.COLOR_BGR2RGB)
    
detector = CourtLineDetector(cfg, model, "keypoint_heatmap_672x448.pth")
keypoints = detector.predict(orig_image)
image = detector.draw_keypoints(orig_image, keypoints)

RuntimeError: Error(s) in loading state_dict for ResNetHeatmap:
	Missing key(s) in state_dict: "backbone.layer1.0.conv3.weight", "backbone.layer1.0.bn3.weight", "backbone.layer1.0.bn3.bias", "backbone.layer1.0.bn3.running_mean", "backbone.layer1.0.bn3.running_var", "backbone.layer1.0.downsample.0.weight", "backbone.layer1.0.downsample.1.weight", "backbone.layer1.0.downsample.1.bias", "backbone.layer1.0.downsample.1.running_mean", "backbone.layer1.0.downsample.1.running_var", "backbone.layer1.1.conv3.weight", "backbone.layer1.1.bn3.weight", "backbone.layer1.1.bn3.bias", "backbone.layer1.1.bn3.running_mean", "backbone.layer1.1.bn3.running_var", "backbone.layer1.2.conv1.weight", "backbone.layer1.2.bn1.weight", "backbone.layer1.2.bn1.bias", "backbone.layer1.2.bn1.running_mean", "backbone.layer1.2.bn1.running_var", "backbone.layer1.2.conv2.weight", "backbone.layer1.2.bn2.weight", "backbone.layer1.2.bn2.bias", "backbone.layer1.2.bn2.running_mean", "backbone.layer1.2.bn2.running_var", "backbo